In [ ]:
import tkinter,math
from time import time
root = tkinter.Tk()
canvas = tkinter.Canvas(root, width=400, height=400)
canvas.pack()
width=400
height=400
coords = [(x,y)]

x, y = 50, 50 # координаты центров
velocity_x, velocity_y = 15, 8 # скорости
r = 25
mass = 20
gravity = 15 * mass


coords_x = []
coords_y = []

time_old = time()

def animate():
    global time_old, x, y, r, velocity_x, velocity_y,coords_x,coords_y
    coords_x.append(x)
    coords_y.append(y)
    coords.append((x,y))

    time_now = time()
    dt = time_now - time_old
    time_old = time_now
    velocity_y+=gravity*dt
    x += velocity_x * dt
    y += velocity_y * dt
    if x < r or x > width - r:
        velocity_x *= -1
    if y < r or y > height - r:
        velocity_y *= -1
    velocity_x*=0.99
    velocity_y*=0.99

    canvas.delete('all')


    for i in range(len(coords_x)-1):
        canvas.create_line(coords[i][0], coords[i][1],coords[i+1][0],coords[i+1][1],fill="green")


    canvas.create_oval(x - r, y - r, x + r, y + r, fill='black', outline='blue')
    canvas.after(50, animate)
animate()
root.mainloop()


spring

In [36]:
root = tkinter.Tk()
W, H = 400,600
x, y = 50,60
L0 = 150 
K = 1
M = 1
A = 0.2
canvas = tkinter.Canvas(root, width=W,height=H)
canvas.configure(bg="black")
canvas.pack()




In [ ]:
from time import time
import tkinter
import math

root = tkinter.Tk()
W = 400
H = 800
canvas = tkinter.Canvas(root, width=W, height=H)
canvas.pack()

N = 15 # число пружин
x, y = [0]*N, [0]*N
velocity = [[0]*N, [0]*N] # скорости
CX, CY, RX, RY = W/2, H/4, W/4, W/4 # начальные параметры пузыря
for i in range(N):
    phi = 2 * math.pi / N * i
    x[i] = CX + RX * math.cos(phi)
    y[i] = CY + RY * math.sin(phi)
gravity = 100 # 200 гравитационное ускорение
L0 = 0.5 * 2 * math.pi * (RX + RY) / 2 / N # начальная длина пружины
K = 0.02 # константа Гука
M = 0.2 / N # масса пружинки
A = 0.01 # сопротивление воздуха

def getNormalVector(p1, p2):
    nx = math.sin(p1) - math.sin(p2)
    ny = -math.cos(p1) + math.cos(p2)
    n = math.sqrt(nx**2 + ny**2)
    nx /= n
    ny /= n
    return (nx, ny)
def GaussShoelace(x, y):
    S = 0
    for i in range(len(x)):
        S += x[i - 1] * y[i] - x[i] * y[i-1]
    return S


BM = 100000 # константа Бойля-Мариотта
def animate(time_old, x, y, gravity, velocity):
    time_now = time()
    dt = time_now - time_old
    new_x, new_y = [0] * N, [0] * N
    canvas.delete("all")
    S = GaussShoelace(x, y)
    P = BM / S # давление
    for i in range(N): # у каждой пружины 2 соседа
        phi1 = math.atan2(y[(i+1) % N] - y[i], x[(i+1) % N] - x[i])
        L1 = math.hypot(x[(i+1) % N] - x[i], y[(i+1) % N] - y[i])
        FH1 = K * (L1 - L0)**2 # закон Гука
        phi2 = math.atan2(y[i-1] - y[i], x[i-1] - x[i])
        L2 = math.hypot(x[i-1] - x[i], y[i-1] - y[i])
        FH2 = K * (L2 - L0)**2 # закон Гука
        nx, ny = getNormalVector(phi1, phi2)

        Fx = FH1 * math.cos(phi1) + FH2 * math.cos(phi2) - A * velocity[0][i] + P * nx
        Fy = M * gravity + FH1 * math.sin(phi1) + FH2 * math.sin(phi2) - A * velocity[1][i] + P * ny
        ax = Fx / M; ay = Fy / M # закон Ньютона 
        velocity[0][i] += ax * dt; velocity[1][i] += ay * dt
        new_x[i] = x[i] + velocity[0][i] * dt
        new_y[i] = y[i] + velocity[1][i] * dt
        canvas.create_line(x[i], y[i], x[i] + 10 * A * velocity[0][i], y[i] + 10 * A * velocity[1][i], fill='red', width=5)
    coords = zip(x,y)
    canvas.create_polygon(*coords,fill='blue')

    for i in range(N):
        x[i] = new_x[i]; y[i] = new_y[i]
        canvas.create_line(x[i], y[i], x[i-1], y[i-1], fill='red', width=5)
    canvas.after(50, animate, time_now, x, y, gravity, velocity)

animate(time(), x, y, gravity, velocity)
root.mainloop()